# run02：Stage 2だけをPRM中心で再学習

**狙い：数学教材を広げる。0.22超えは未検証です。**
Stage 1はrun01のadapterを再利用。Stage 2は新規LoRAを作り、PRM800K 1,500件＋GSM8K 500件を解説全文で1周学習します。
run01のStage 2は読み込まず、上書きもしません。学習率5e-5、rank 8 / alpha 16、実効batch 16、系列長2048を維持します。
全文が収まらない例は機械的に除外し、解説の途中で切りません。lossは全非paddingトークンが対象です。

## 実行手順：新しいColab T4で、この1冊だけをアップロード

1. `PHASE="setup"` → **すべて実行** → セッション再起動。
2. `PHASE="merge"` → **すべて実行** → セッション再起動。ランタイムは削除しない。
3. `PHASE="train"` → **すべて実行**。終了すると新しいStage 2がDriveに保存されます。

Driveに `geniac_workspace/adapters/t4_20261013_run01_stage1/` が必要です。
別アカウントなら同じパスに元のStage 1フォルダを置いてください。配布教材はこのノートブックが取得します。
停止した学習はセッションのみ再起動し、同じ設定で `PHASE="train"` を再実行すると最新checkpointから再開します。
ランタイム削除後はsetupとmergeも再実行してください。最初の50step以前にはcheckpointがありません。

固定した評価用800問は、同梱した問題ハッシュで学習から除外します。問題・解答プレビューは表示しません。

In [ ]:
# [config]
PHASE = "setup"                   # setup → 再起動 → merge → 再起動 → train
RUN_ID = "t4_20261013_run02_prmfull"
WORKSPACE = "/content/drive/MyDrive/geniac_workspace"
STAGE1_DIR = WORKSPACE + "/adapters/t4_20261013_run01_stage1"
STAGE2_DIR = WORKSPACE + "/adapters/" + RUN_ID + "_stage2"
MERGED_STAGE1 = "/content/run02_stage1_merged"
RUN_DIR = WORKSPACE + "/experiments/" + RUN_ID
COUNTS = {"prm800k": 1500, "gsm8k": 500}
MAX_LENGTH = 2048
EPOCHS = 1.0
LEARNING_RATE = 5e-5
LORA_R, LORA_ALPHA = 8, 16
GRAD_ACCUM = 16
SEED = 42

In [ ]:
# [guard-and-install]
import sys, subprocess, json, os, hashlib, time, gc
from pathlib import Path
if not Path('/content').is_dir():
    raise RuntimeError('このnotebookはColab CUDA用です。T4へ接続してください。')
import google.colab
if PHASE not in ('setup', 'merge', 'train'):
    raise ValueError('PHASEを確認してください。')
if PHASE == 'setup':
    with open('/content/run02_install.log', 'w') as log:
        result = subprocess.run([sys.executable, '-m', 'pip', 'install',
                                 'unsloth', 'trl', 'bitsandbytes', 'datasets', 'peft', 'accelerate'],
                                stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError('導入失敗。/content/run02_install.log を確認してください。')
    print('setup完了。セッション再起動 → PHASE="merge" → すべて実行。')

## 固定情報と教材準備の関数
評価問題の本文は含めず、除外用ハッシュ・許可教材のrevision・元adapterのハッシュだけを同梱しています。

In [ ]:
# [reference]
REFERENCE = {'base': {'commit': '906bfd4b4dc7f14ee4320094d8b41684abff8539', 'model': 'Qwen/Qwen3-4B-Base'},
 'hashes': ['...'],
 'normalization': 'NFKC + whitespace collapse + sha256(utf8)',
 'sources': {'gsm8k': {'config': 'main',
                       'id': 'openai/gsm8k',
                       'license': 'MIT',
                       'revision': '740312add88f781978c0658806c59bc2815b9866',
                       'split': 'train'},
             'prm800k': {'file_sha256': '90d96daeac3fe343ebb1e22ce93dd99690f75983e957f88de42f87cffe1e8076',
                         'license': 'MIT',
                         'revision': '7ecc794703b2877f63226f2477a49b34f9b25163',
                         'url': 'https://media.githubusercontent.com/media/openai/prm800k/7ecc794703b2877f63226f2477a49b34f9b25163/prm800k/math_splits/train.jsonl'}},
 'split_sha256': '52d27952511b10494d31620fca969e40a1a3986bb1d3bea28ef5f09d9e96ac55',
 'stage1_files': {'adapter_config.json': '6a70b2f74435f0d8dcb23e24af8ab52f3fe0be09927797468b6e97eac1ee12f1',
                  'adapter_model.safetensors': '8daee3c3c9a1627263b746e366938ddc53232f75dcf1f416c135c58bf042a709'}}
CHAT_TEMPLATE = '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0].role == \'system\' %}\n        {{- messages[0].content + \'\\n\\n\' }}\n    {%- endif %}\n    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0].content + \'<|im_end|>\\n\' }}\n    {%- endif %}\n{%- endif %}\n{%- for message in messages %}\n    {%- if message.content is string %}\n        {%- set content = message.content %}\n    {%- else %}\n        {%- set content = \'\' %}\n    {%- endif %}\n    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}\n        {{- \'<|im_start|>\' + message.role + \'\\n\' + content + \'<|im_end|>\' + \'\\n\' }}\n    {%- elif message.role == "assistant" %}\n        {{- \'<|im_start|>\' + message.role + \'\\n\' + content }}\n        {%- if message.tool_calls %}\n            {%- for tool_call in message.tool_calls %}\n                {%- if (loop.first and content) or (not loop.first) %}\n                    {{- \'\\n\' }}\n                {%- endif %}\n                {%- if tool_call.function %}\n                    {%- set tool_call = tool_call.function %}\n                {%- endif %}\n                {{- \'<tool_call>\\n{"name": "\' }}\n                {{- tool_call.name }}\n                {{- \'", "arguments": \' }}\n                {%- if tool_call.arguments is string %}\n                    {{- tool_call.arguments }}\n                {%- else %}\n                    {{- tool_call.arguments | tojson }}\n                {%- endif %}\n                {{- \'}\\n</tool_call>\' }}\n            {%- endfor %}\n        {%- endif %}\n        {{- \'<|im_end|>\\n\' }}\n    {%- elif message.role == "tool" %}\n        {%- if loop.first or (messages[loop.index0 - 1].role != "tool") %}\n            {{- \'<|im_start|>user\' }}\n        {%- endif %}\n        {{- \'\\n<tool_response>\\n\' }}\n        {{- content }}\n        {{- \'\\n</tool_response>\' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}\n            {{- \'<|im_end|>\\n\' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\n\' }}\n{%- endif %}'
RECIPE_CODE_SHA256 = '6cef7ce1592666da72823d79db3cb1d8a5a23368777ec58cf568cead8e0aabbd'

In [ ]:
# [recipe-functions]
"""Mechanical preparation for run02. No model inference or semantic selection."""
import hashlib
import json
import re
import urllib.request
from collections import Counter


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def question_hash(text):
    import unicodedata
    return hashlib.sha256(' '.join(unicodedata.normalize('NFKC', text).split()).encode()).hexdigest()


def boxed_answer(text):
    matches = list(re.finditer(r'\\boxed\s*\{', text))
    if not matches:
        return None
    start = matches[-1].end()
    depth = 1
    for i in range(start, len(text)):
        if i and text[i - 1] == '\\':
            continue
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i].strip()
    return None


def full_solution(solution, answer):
    solution = solution.strip()
    if boxed_answer(solution) is not None:
        return solution
    return solution + '\n\nTherefore, $\\boxed{' + (boxed_answer(answer) or answer.strip()) + '}$.'


def fetch_allowed_records(sources):
    from datasets import load_dataset
    gsm = sources['gsm8k']
    assert gsm['id'] == 'openai/gsm8k' and gsm['config'] == 'main' and gsm['split'] == 'train'
    prm = sources['prm800k']
    url = ('https://media.githubusercontent.com/media/openai/prm800k/'
           + prm['revision'] + '/prm800k/math_splits/train.jsonl')
    assert prm['url'] == url
    with urllib.request.urlopen(url, timeout=120) as f:
        data = f.read()
    if hashlib.sha256(data).hexdigest() != prm['file_sha256']:
        raise RuntimeError('PRM800Kの取得内容が固定版と異なります。')
    rows = []
    for line in data.decode().splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        if all(isinstance(record.get(k), str) and record[k].strip()
               for k in ('problem', 'solution', 'answer')):
            rows.append({k: record[k].strip() for k in ('problem', 'solution', 'answer')}
                        | {'dataset': 'prm800k'})
    for row in load_dataset('openai/gsm8k', 'main', split='train', revision=gsm['revision']):
        solution, sep, answer = row['answer'].partition('####')
        if sep and row['question'].strip() and solution.strip() and answer.strip():
            rows.append({'problem': row['question'].strip(),
                         'solution': re.sub(r'<<[^>]*>>', '', solution).strip(),
                         'answer': answer.strip(), 'dataset': 'gsm8k'})
    return rows


def build_recipe(records, tokenizer, heldout_hashes, counts, max_length=2048, seed=42):
    """Fixed hash ordering, dedup, whole-example length filtering, no truncation."""
    blocked = set(heldout_hashes)
    seen, stats, pools = set(), Counter(), {name: [] for name in counts}
    if tokenizer.eos_token_id is None:
        raise RuntimeError('EOS token is required.')
    ordered = sorted(records, key=lambda r: (r['dataset'] != 'prm800k',
                      stable_hash([seed, question_hash(r['problem'])])))
    for row in ordered:
        key, name = question_hash(row['problem']), row['dataset']
        if key in blocked:
            stats[name + '_heldout_excluded'] += 1
            continue
        if key in seen:
            stats[name + '_duplicate_excluded'] += 1
            continue
        seen.add(key)
        target = full_solution(row['solution'], row['answer'])
        ids = tokenizer.apply_chat_template([{'role': 'user', 'content': row['problem']},
                                             {'role': 'assistant', 'content': target}], tokenize=True)
        if ids[-1] != tokenizer.eos_token_id:
            ids.append(tokenizer.eos_token_id)
        if len(ids) > max_length:
            stats[name + '_over_length_excluded'] += 1
            continue
        pools[name].append((dict(row, problem_hash=key), ids))
    selected = []
    for name, n in counts.items():
        stats[name + '_eligible'] = len(pools[name])
        if len(pools[name]) < n:
            raise RuntimeError('全文が収まる教材が不足: ' + name)
        selected.extend(pools[name][:n])
    selected.sort(key=lambda pair: stable_hash(['shuffle', seed, pair[0]['problem_hash']]))
    raw = [r for r, ids in selected]
    tokenized = [{'input_ids': ids, 'attention_mask': [1] * len(ids)} for r, ids in selected]
    selected_hashes = [r['problem_hash'] for r in raw]
    assert len(selected_hashes) == len(set(selected_hashes)) and not set(selected_hashes) & blocked
    stats.update({'selected': len(raw), 'tokens': sum(len(x['input_ids']) for x in tokenized),
                  'max_tokens': max(len(x['input_ids']) for x in tokenized),
                  'all_end_with_eos': all(x['input_ids'][-1] == tokenizer.eos_token_id for x in tokenized)})
    return raw, tokenized, dict(stats)

In [ ]:
# [io-functions]
def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2)+'\n')
    os.replace(tmp, path)

def versions():
    from importlib.metadata import version
    return {k: version(k) for k in ['torch','transformers','trl','peft','unsloth','bitsandbytes','datasets']}

def saved_tokenizer(path):
    from transformers import AutoTokenizer
    from importlib.metadata import version
    cfg = read_json(Path(path)/'tokenizer_config.json')
    kwargs = {}
    if int(version('transformers').split('.')[0]) < 5 and isinstance(cfg.get('extra_special_tokens'), list):
        kwargs = {'extra_special_tokens': {}, 'additional_special_tokens': cfg['extra_special_tokens']}
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=False, **kwargs)
    for token_id, token in cfg.get('added_tokens_decoder', {}).items():
        assert tok.convert_tokens_to_ids(token['content']) == int(token_id), 'token IDが変わっています。'
    return tok

In [ ]:
# [workspace]
if PHASE != 'setup':
    # Unsloth must precede transformers/peft/trl imports in the training process.
    if PHASE == 'train':
        from unsloth import FastLanguageModel
    import torch
    if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
        raise RuntimeError('ランタイムをT4 GPUへ変更してください。')
    from google.colab import drive
    drive.mount('/content/drive')
    if Path(STAGE2_DIR).exists():
        raise RuntimeError('同名のStage 2保存先が既にあります。上書きしません。完了済みか確認してください。')
    for filename, expected in REFERENCE['stage1_files'].items():
        path = Path(STAGE1_DIR)/filename
        if not path.is_file() or file_sha(path) != expected:
            raise RuntimeError('run01 Stage 1と異なる/不足するファイル: '+filename)
    MERGE_SIGNATURE = {'base': REFERENCE['base'], 'stage1': REFERENCE['stage1_files'],
                       'template_sha256': hashlib.sha256(CHAT_TEMPLATE.encode()).hexdigest()}
    Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

## Stage 1を復元（mergeのみ）
公式Baseに保存済みStage 1を統合します。これは学習ではありません。元の4bit学習直後に保存したモデルとの数値的同一性は仮定しません。

In [ ]:
# [restore-stage1]
if PHASE == 'merge':
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    marker = Path(MERGED_STAGE1)/'run02_merge.json'
    if Path(MERGED_STAGE1).exists():
        if not marker.exists() or read_json(marker)['signature'] != MERGE_SIGNATURE:
            raise RuntimeError('未完了/別構成のMERGED_STAGE1です。空の別パスを指定してください。')
        print('同じStage 1の復元記録を再利用します。')
    else:
        model = AutoModelForCausalLM.from_pretrained(
            REFERENCE['base']['model'], revision=REFERENCE['base']['commit'],
            dtype=torch.float16, device_map='cuda', low_cpu_mem_usage=True)
        model = PeftModel.from_pretrained(model, STAGE1_DIR).merge_and_unload(safe_merge=True)
        tokenizer = saved_tokenizer(STAGE1_DIR)
        if tokenizer.chat_template != CHAT_TEMPLATE:
            raise RuntimeError('Stage 1 tokenizerと配布templateが一致しません。')
        model.save_pretrained(MERGED_STAGE1, safe_serialization=True, max_shard_size='2GB')
        tokenizer.save_pretrained(MERGED_STAGE1)
        files = {p.name: file_sha(p) for p in sorted(Path(MERGED_STAGE1).iterdir()) if p.is_file()}
        save_json(marker, {'signature': MERGE_SIGNATURE, 'files': files})
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    print('merge完了。セッションのみ再起動 → PHASE="train" → すべて実行。')

## 教材の選択（trainのみ）
PRM 1,500件＋GSM 500件を固定seedで機械的に選びます。解説全文とEOSを含めて2048以内の例のみ採用します。

In [ ]:
# [prepare-training-data]
if PHASE == 'train':
    marker = Path(MERGED_STAGE1)/'run02_merge.json'
    if not marker.exists() or read_json(marker)['signature'] != MERGE_SIGNATURE:
        raise RuntimeError('PHASE="merge"を先に実行してください。')
    merged_record = read_json(marker)
    for name, expected in merged_record['files'].items():
        if not (Path(MERGED_STAGE1)/name).is_file() or file_sha(Path(MERGED_STAGE1)/name) != expected:
            raise RuntimeError('Stage 1統合ファイルが欠損/変更されています。')
    tokenizer = saved_tokenizer(MERGED_STAGE1)
    raw, tokenized, data_stats = build_recipe(fetch_allowed_records(REFERENCE['sources']), tokenizer,
        REFERENCE['hashes'], COUNTS, max_length=MAX_LENGTH, seed=SEED)
    signature = {'run_id': RUN_ID, 'merge': MERGE_SIGNATURE,
                 'merged_files': merged_record['files'], 'versions': versions(),
                 'recipe_code_sha256': RECIPE_CODE_SHA256, 'heldout_split': REFERENCE['split_sha256'],
                 'selected_ids': [r['problem_hash'] for r in raw],
                 'config': {'counts': COUNTS, 'max_length': MAX_LENGTH, 'epochs': EPOCHS,
                            'learning_rate': LEARNING_RATE, 'rank': LORA_R, 'alpha': LORA_ALPHA,
                            'grad_accum': GRAD_ACCUM, 'seed': SEED, 'loss': 'all_nonpadding',
                            'compression': 'none', 'explicit_eos': True}}
    signature_path = Path(RUN_DIR)/'training_signature.json'
    if signature_path.exists() and read_json(signature_path) != signature:
        raise RuntimeError('以前の学習と構成が異なります。別RUN_IDを設定してください。')
    save_json(signature_path, signature)
    cache = Path(WORKSPACE)/'data_cache'/(RUN_ID+'_'+str(sum(COUNTS.values()))+'.jsonl')
    cache.parent.mkdir(parents=True, exist_ok=True)
    data_text = ''.join(json.dumps(r, ensure_ascii=False)+'\n' for r in raw)
    if cache.exists() and cache.read_text() != data_text:
        raise RuntimeError('同名の学習キャッシュが別内容です。')
    cache.write_text(data_text)
    save_json(Path(RUN_DIR)/'data_summary.json', data_stats)
    print('学習教材:', COUNTS, '/ 集計:', data_stats)
    print('学習cache:', str(cache))

## 新しいStage 2を学習
Stage 1統合モデルを4bitで読み込み、新しいLoRAを追加します。回答のみlossなどの追加変更は今回は行いません。

In [ ]:
# [new-stage2-model]
if PHASE == 'train':
    if 'model' in globals():
        raise RuntimeError('モデルが残っています。セッション再起動後、trainを再実行してください。')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MERGED_STAGE1, max_seq_length=MAX_LENGTH,
        dtype=torch.float16, load_in_4bit=True, use_exact_model_name=True)
    assert tokenizer.chat_template == CHAT_TEMPLATE, '学習templateが変わっています。'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA,
        lora_dropout=0.0, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none', use_gradient_checkpointing='unsloth', random_state=SEED)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not trainable:
        raise RuntimeError('学習対象のパラメータがありません。')
    print('学習対象パラメータ:', trainable)

In [ ]:
# [trainer-and-checks]
if PHASE == 'train':
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    class CheckFiniteLoss(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            import math
            for key in ('loss', 'grad_norm'):
                value = (logs or {}).get(key)
                if isinstance(value, (int, float)) and not math.isfinite(value):
                    raise RuntimeError('非有限の'+key+'を検出しました。学習を停止します。')
    cfg = SFTConfig(output_dir=RUN_DIR+'/checkpoints', num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE, per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, lr_scheduler_type='cosine',
        max_length=MAX_LENGTH, packing=False, completion_only_loss=False,
        bf16=False, fp16=True, optim='paged_adamw_8bit', logging_steps=5, seed=SEED,
        save_strategy='steps', save_steps=50, save_total_limit=2, save_only_model=False,
        report_to='none')
    trainer = SFTTrainer(model=model, processing_class=tokenizer,
        train_dataset=Dataset.from_list(tokenized), args=cfg, callbacks=[CheckFiniteLoss()])
    # Check real collated labels rather than assuming a config flag worked.
    for i in (0, len(tokenized)-1):
        example = trainer.train_dataset[i]
        batch = trainer.data_collator([example])
        mask = batch['attention_mask'].bool()
        if not torch.equal(batch['input_ids'][mask], batch['labels'][mask]):
            raise RuntimeError('想定外のlossマスクを検出しました。')
        if not torch.all(batch['labels'][~mask] == -100):
            raise RuntimeError('paddingのlossマスクが不正です。')
        n = int(mask.sum())
        if (batch['input_ids'][mask].tolist() != tokenized[i]['input_ids']
                or batch['input_ids'][0,n-1].item() != tokenizer.eos_token_id):
            raise RuntimeError('データの切詰め/EOS欠落を検出しました。')
    print('実際のlabelsを確認：非padding全体を学習、全文/EOSを保持。')

In [ ]:
# [train-and-save]
if PHASE == 'train':
    from transformers.trainer_utils import get_last_checkpoint
    checkpoint_dir = Path(RUN_DIR)/'checkpoints'
    checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None
    print('再開checkpoint:', checkpoint or 'なし（新規学習）')
    t0 = time.monotonic()
    result = trainer.train(resume_from_checkpoint=checkpoint)
    # Do not mark a partial/early-stopped run as a final adapter.
    if trainer.state.epoch is None or trainer.state.epoch < EPOCHS - 1e-3:
        raise RuntimeError('予定epoch未完了。最終adapterとして保存しません。')
    if Path(STAGE2_DIR).exists():
        raise RuntimeError('保存先が既にあります。上書きしません。')
    model.save_pretrained(STAGE2_DIR)
    tokenizer.save_pretrained(STAGE2_DIR)
    report = {'complete': True, 'run_id': RUN_ID, 'stage1_dir': STAGE1_DIR,
              'stage2_dir': STAGE2_DIR, 'global_step': trainer.state.global_step,
              'epoch': trainer.state.epoch, 'metrics': result.metrics,
              'this_session_seconds': time.monotonic()-t0, 'data_stats': data_stats,
              'signature_sha256': stable_hash(signature),
              'stage2_weights_sha256': file_sha(Path(STAGE2_DIR)/'adapter_model.safetensors')}
    save_json(Path(RUN_DIR)/'training_summary.json', report)
    print('Stage 2学習完了:', STAGE2_DIR)
    print('集計:', RUN_DIR+'/training_summary.json')

## 学習が終わったら

新しいT4推論ランタイムで、既存の `evaluate_dev.ipynb` または配布の推論notebookを使用します。
**Stage 1はrun01、Stage 2だけrun02** にします。

```python
STAGE1_SLUG = "t4_20261013_run01_stage1"
STAGE2_SLUG = "t4_20261013_run02_prmfull_stage2"
RUN_ID = "t4_20261013_run02_prmfull"
```

開発評価側では `STAGE2_ADAPTER_DIR` を新しい保存先へ変え、`EVAL_RUN_ID`も別名にします。
`TRAIN_CACHE_SPECS` に今回の `data_cache/t4_20261013_run02_prmfull_2000.jsonl`（2,000件）を追加してください。
固定splitは800問の除外ハッシュに対応する元のものを使います。ローカルの `local_eval/split_v1.json` と
`heldout_problem_hashes.json` をDriveの `geniac_workspace/dev_eval/` に置けば同じ評価問題を使えます。

公式提出は、配布推論コードで全問生成した未編集のJSONを使用します。このnotebookは提出しません。
先に開発比較する場合、run01とrun02を同じT4/vLLM条件で測ってください。学習lossの低下だけでは性能改善を判断できません。

初回の学習時間は未実測です。ローカル事前準備では平均約264トークン/件、合計約53万トークンでした。
元のGSM8Kより長いため、run01の18.5分をそのまま当てはめないでください。

参考：[TRL SFTTrainer](https://huggingface.co/docs/trl/en/sft_trainer)、
[Unsloth checkpoint再開](https://unsloth.ai/docs/basics/finetuning-from-last-checkpoint)。